# Notebook 05b — Red-Light Violation

**Objective.** Build a red-light rule and measure its **precision and recall on real violations**, on clips that played no part in shaping it.

**Why this notebook exists.** Notebook 05 could not test a red-light rule: on `video-test.mp4` it is unverified which traffic the visible signal governs, and no stop line is verifiable. This notebook uses footage where the signal, the stop line and real violations are all visible. As in Notebook 05, the output is a **candidate** for human review, never a verdict.

**Input.**
- `data/raw/rlr_occasions.mp4` — a screen recording of a public YouTube compilation of red-light running (18 clips of one junction, 1280x720, 30 FPS). It stays local and is not committed.
- `configs/scenes/rlr_occasions.json` — the stop line, lane dividers and signal positions of each clip.
- `data/annotations/rlr_occasions_labels.json` — hand-made labels, the matching criterion, and their known limits.
- `data/traffic_video_modified.mp4` — **only** for a logic test in section 8: its signal is a synthetic overlay.

**Output**, in `outputs/05b_red_light_violation/` (git-ignored):
- `rlr_occasions.red_light.candidates.json` — candidate events with status `pending`, every crossing record, and the scores.
- `violation_clips/` — **one video per candidate event**, with original pixels, and `manifest.json` saying where the vehicle is in each frame. This is the input to plate reading in Notebook 06.
- `review/` — annotated clips and a sheet for a person to check the labels.
- `logic_test_clips/` — the same output for the synthetic-signal video, kept apart from real events.
- `function_output/` — the results of `detect_red_light_video`, the one function that joins these steps (section 12).
- `rule_v2/` — the same outputs with rule version 2 (section 13), the input to Notebook 06.
- `tracks.json`, `signals.json`, `overlay_run.json` — caches of the slow steps.

**How the evaluation was kept honest.** The order of the work is visible in the git history:

1. The rule was shaped by inspecting its mistakes on three **development clips** (B, F, H), then committed.
2. The other 15 **evaluation clips** were labelled from frames showing no tracker or rule output, and the labels and matching criterion were committed.
3. Only then was the rule run on those clips.

**Images stay local.** The footage shows real vehicles and belongs to a third party, so a git filter removes every image output from the committed copy of this notebook; run it locally to see them. Nothing is blurred, in the images or in the clips, because the next step has to read the plates. This notebook reads no plates itself.

## 0. Setup

In [1]:
import csv
import json
import logging
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

from traffic_violation_detection.evidence import CLIP_SETTINGS, clip_frame_range, cut_event_clip, write_clip
from traffic_violation_detection.red_light import (
    AT_BORDER,
    CANDIDATE,
    COMPLIANT,
    ENDED,
    OUT_OF_RANGE,
    RIGHT_TURN,
    RULE as RULE_V2,
    RULE_V1 as RULE,  # Sections 1 to 12 keep version 1, the rule that was frozen and evaluated.
    STOPPED,
    UNKNOWN_SIGNAL,
    build_event,
    deduplicate,
    evaluate_track,
    lane_of,
    line_y,
    load_red_light_scene,
    match_labels_to_crossings,
    read_group_state,
    read_lamp_state,
    score_candidates,
    smooth_states,
)
from traffic_violation_detection.scene import frame_to_reference, reference_to_frame, similarity_matrix
from traffic_violation_detection.video import iter_frames, read_frame_at, read_video_metadata

%config InlineBackend.figure_formats = ["jpeg"]

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found in {start} or any parent directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
VIDEO = PROJECT_ROOT / "data" / "raw" / "rlr_occasions.mp4"
OVERLAY_VIDEO = PROJECT_ROOT / "data" / "traffic_video_modified.mp4"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "05b_red_light_violation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

meta = read_video_metadata(VIDEO)
FPS, W, H = meta["fps"], meta["width"], meta["height"]
scene = load_red_light_scene(PROJECT_ROOT / "configs" / "scenes" / "rlr_occasions.json", W, H)
labels = json.loads((PROJECT_ROOT / "data" / "annotations" / "rlr_occasions_labels.json").read_text())

VIDEO_ID = scene["scene_id"]
CLIPS = scene["clips"]
LANE_ANGLE = scene["lane_direction_deg"]
MATCH = labels["matching"]
DEVELOPMENT = [n for n, c in CLIPS.items() if c["role"] == "development"]
EVALUATION = [n for n, c in CLIPS.items() if c["role"] == "evaluation"]
ALL_LABELS = {**labels["clips"], **labels["development_clips"]}
assert set(ALL_LABELS) == set(CLIPS)
print(f"{W}x{H}, {FPS} FPS, {meta['reported_frame_count']} frames")
print(f"development clips: {DEVELOPMENT}\nevaluation clips:  {EVALUATION}")
print("labels written by:", labels["annotator"]["kind"], "| reviewed by a person:", labels["annotator"]["reviewed_by_a_person"])

1280x720, 30.0 FPS, 5359 frames
development clips: ['B', 'F', 'H']
evaluation clips:  ['A', 'C', 'D', 'E', 'G', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R']
labels written by: ai_assistant | reviewed by a person: False


In [3]:
def show(image, title=None, dpi=100):
    height, width = image.shape[:2]
    figure, axis = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.axis("off")
    if title:
        axis.set_title(title, fontsize=11, loc="left")
    plt.show()


def grid(tiles, columns):
    blank = np.zeros_like(tiles[0])
    tiles = tiles + [blank] * (-len(tiles) % columns)
    return np.vstack([np.hstack(tiles[i:i + columns]) for i in range(0, len(tiles), columns)])

## 1. The data

The compilation shows one junction, seen from behind the traffic. Between clips the camera's framing changes by tens of pixels, and within a clip it moved by at most 2.8 px when measured, so each clip gets **its own geometry** and no registration.

Lighting varies: daytime, colour night, and infrared. Clip H is entirely monochrome, and clip I starts monochrome, so colour cannot be read there.

In [4]:
print(f"{'clip':<5}{'role':<13}{'lighting':<10}{'seconds':>8}  {'signal read by':<15}{'labelled violations':>20}")
for name, clip in CLIPS.items():
    count = sum(e["label"] == "violation" for e in ALL_LABELS[name]["events"])
    print(f"{name:<5}{clip['role']:<13}{clip['lighting']:<10}{(clip['end'] - clip['start']) / FPS:>8.1f}  {clip['signals']['method']:<15}{count:>20}")

clip role         lighting   seconds  signal read by  labelled violations
A    evaluation   day            9.3  lamps                             1
B    development  day           18.5  lamps                             1
C    evaluation   night          6.9  head                              2
D    evaluation   day            9.0  lamps                             1
E    evaluation   night          9.0  head                              1
F    development  night          5.9  head                              1
G    evaluation   night         16.5  head                              1
H    development  infrared      20.9  head                              5
I    evaluation   night          4.5  head                              0
J    evaluation   night          6.8  head                              1
K    evaluation   night         12.0  head                              1
L    evaluation   night          4.3  head                              1
M    evaluation   night          5.3  

## 2. Scene geometry

The configuration declares, for each clip:

- **Stop line:** a polyline through the ends of the lane markings. The painted line itself is faint or absent in much of the footage.
- **Lane dividers:** used to tell the left-turn lane, governed by the left signal head, from the rest.
- **Signal positions:** at night, a box around each head, read by the colour of its glow. By day, the position of each lamp, because a yellow backplate misleads a whole-head reading.

The images are each clip's median, so moving vehicles disappear. Green is the stop line, magenta the dividers, orange the left signal group, and yellow the through group.

In [5]:
def clip_median(name):
    path = OUTPUT_DIR / "medians" / f"{name}.jpg"
    if not path.exists():
        clip = CLIPS[name]
        frames = [r["image"] for r in iter_frames(VIDEO, clip["start"], clip["end"], step=6)]
        path.parent.mkdir(exist_ok=True)
        cv2.imwrite(str(path), np.median(np.stack(frames), axis=0).astype(np.uint8))
    return cv2.imread(str(path))


def draw_geometry(image, clip):
    out = image.copy()
    points = np.array(clip["stop_line"], np.int32)
    cv2.polylines(out, [points], False, (0, 255, 0), 2, cv2.LINE_AA)
    for divider in clip["lane_dividers"]:
        y = int(line_y(clip["stop_line"], divider))
        cv2.line(out, (divider, y - 16), (divider, y + 16), (255, 0, 255), 2)
    signals = clip["signals"]
    for group, colour in (("left", (0, 140, 255)), ("through", (0, 255, 255))):
        for head in signals[group]:
            if signals["method"] == "head":
                cv2.rectangle(out, tuple(head[:2]), tuple(head[2:]), colour, 2)
            else:
                for x, y in head.values():
                    cv2.circle(out, (x, y), 7, colour, 2, cv2.LINE_AA)
    return out


tiles = []
for name, clip in CLIPS.items():
    tile = cv2.resize(draw_geometry(clip_median(name), clip), (426, 240), interpolation=cv2.INTER_AREA)
    cv2.putText(tile, f"{name} ({clip['role'][:3]})", (8, 228), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
    tiles.append(tile)
show(grid(tiles, 3))

**Assumptions the geometry rests on**, also recorded in the configuration:

In [6]:
for assumption in scene["assumptions"]:
    print("-", assumption)

- Coordinates are pixels of each clip; the camera is stable within a clip but its framing differs between clips.
- Mast heads 2 and 3 govern the through lanes; head 1 governs the left-turn lane, left of the first lane divider. Supported by clip M, where they differ, but not confirmed for every clip.
- The lane right of the last divider is a right-turn lane. Right turns on red may be legal in this jurisdiction, so they are not flagged.
- A vehicle's front wheels are estimated at 30% of the box height above the box bottom, since vehicles are seen from behind.
- Clips use the role 'development' if they were used to shape the rule, and 'evaluation' if they were kept out of that work.


## 3. The rule

The code is in `src/traffic_violation_detection/red_light.py`, with tests in `tests/test_red_light.py`. Its thresholds:

In [7]:
for key, value in RULE.items():
    print(f"{key:<24}{value}")

name                    red_light
version                 1
entry_offset            0.3
pass_margin             0.5
min_track_frames        8
smoothing_frames        5
min_frames_to_decide    15
still_box_heights       0.15
min_heading_px          60
turn_angle_deg          35
border_margin           3
duplicate_seconds       1.0
duplicate_px            120
join_fragments          False
join_max_gap_frames     6
join_max_jump_heights   0.1
join_max_size_ratio     1.25
estimate_hidden_entry   False
hidden_fit_frames       10
hidden_max_frames       30
hidden_margin_frames    13
hidden_min_speed        0.01


**A crossing's verdict**, checked in this order:

| Verdict | Condition | Flagged? |
|---|---|---|
| `not_evaluable_border` | The box touches the image border at entry, so its position is unreliable | no |
| `not_evaluable_occluded_line` | The crossing is under a video overlay | no |
| `not_evaluable_signal` | The signal of its lane is unreadable or its heads disagree | no |
| `signal_not_red` | The signal is yellow or green at entry | no |
| `right_turn_on_red` | Its movement after the line turns right of the lane direction, which may be legal | no |
| `candidate` | Its rear passes the line by half a box height | **yes** |
| `not_evaluable_ended` | Its track or clip ends too soon to tell, or it is still moving when its track ends | no |
| `stopped_at_line` | It stays still with its rear before that point | no |

**Each condition exists because of a mistake found on the development clips or in testing:**

- *Stopped on the line.* A car with its front over the line was flagged. The rear must now pass by half a box height (clip F).
- *Boxes at the border.* A cropped box gave a false position (clip F).
- *Track fragments.* A pickup stopped at the line, then moved on under a new track ID, so its entry was never seen. A track that starts with the front already past the line is now judged when its rear crosses (clip B).
- *Yellow backplates.* By day, a head's yellow backplate read as a yellow lamp, so each lamp is now read separately (clip B).
- *Diagonal lanes.* Lanes run at about −24° in the image, so turns are measured against the lane and not the image axes.
- *Moving at the end of a track.* A vehicle still moving when its track ended was called stopped. A unit test exposed this, and it is now `not_evaluable_ended`.

**The front of a vehicle** is estimated at 30% of the box height above the box bottom, since vehicles are seen from behind.

## 4. Detection and tracking

The baseline from Notebook 03: `yolo26m` at `imgsz=1280`, with OC-SORT using the low-confidence association of ByteTrack. The tracker is reset for each clip, because a vehicle cannot continue across a cut. Results are cached.

In [8]:
COCO_VEHICLES = {1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
TRACKS_PATH = OUTPUT_DIR / "tracks.json"


def load_detector_and_tracker_factory():
    import boxmot
    import boxmot.utils  # noqa: F401  Imported so that the logger below exists.
    import torch
    from ultralytics import YOLO

    assert torch.cuda.is_available(), "Detection needs the GPU server."
    logging.getLogger("boxmot").setLevel(logging.WARNING)
    torch.set_num_threads(16)
    model = YOLO(str(PROJECT_ROOT / "weights" / "yolo26m.pt"))
    return model, lambda: boxmot.OcSort(boxmot.OcSortConfig(use_byte=True))


def detect(model, image):
    result = model.predict(image, imgsz=1280, conf=0.10, iou=0.7, classes=list(COCO_VEHICLES), device=0, verbose=False)[0]
    return np.c_[result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy(), result.boxes.cls.cpu().numpy()].astype(np.float32)


if TRACKS_PATH.exists():
    tracks = json.loads(TRACKS_PATH.read_text())
else:
    model, new_tracker = load_detector_and_tracker_factory()
    tracks = {}
    for name, clip in CLIPS.items():
        tracker, clip_tracks = new_tracker(), {}
        for record in iter_frames(VIDEO, clip["start"], clip["end"]):
            detections = detect(model, record["image"])
            output = tracker.update(detections, record["image"])
            for row in output if len(output) else []:
                clip_tracks.setdefault(str(int(row[4])), []).append(
                    [record["frame_index"], *[round(float(v), 1) for v in row[:4]], int(detections[int(row[7]), 5]), round(float(row[5]), 3)]
                )
        tracks[name] = clip_tracks
    TRACKS_PATH.write_text(json.dumps(tracks))

print(f"{'clip':<6}{'tracks':>7}{'>= 1 s':>8}")
for name in CLIPS:
    print(f"{name:<6}{len(tracks[name]):>7}{sum(len(v) >= FPS for v in tracks[name].values()):>8}")

clip   tracks  >= 1 s
A          30      26
B          36      20
C           5       3
D          33      32
E           1       1
F          11       9
G           4       4
H           8       8
I          13      11
J           1       1
K           3       3
L           2       1
M           2       2
N           3       2
O           2       2
P           1       1
Q           8       3
R          22      22


## 5. Reading the signal

The signal is read from pixels in every frame, then smoothed with a 7-frame majority vote. A group's state is `conflict` if its heads disagree, and `unknown` if none is readable. Neither can produce a candidate.

Each row below is one clip: the upper bar is the through signal, the lower bar the left signal.

In [9]:
SIGNALS_PATH = OUTPUT_DIR / "signals.json"
if SIGNALS_PATH.exists():
    signals = {n: {g: {int(k): s for k, s in v.items()} for g, v in c.items()} for n, c in json.loads(SIGNALS_PATH.read_text()).items()}
else:
    signals = {}
    for name, clip in CLIPS.items():
        raw = {"through": [], "left": []}
        for record in iter_frames(VIDEO, clip["start"], clip["end"]):
            for group in raw:
                raw[group].append(read_group_state(record["image"], clip["signals"]["method"], clip["signals"][group]))
        signals[name] = {g: {clip["start"] + i: s for i, s in enumerate(smooth_states(v))} for g, v in raw.items()}
    SIGNALS_PATH.write_text(json.dumps(signals))

In [10]:
STATE_STYLE = {
    "red": ("#C62828", None, "R"),
    "yellow": ("#D69A00", None, "Y"),
    "green": ("#00897B", None, "G"),
    "unknown": ("#D9D9D9", "////", ""),
    "conflict": ("#7B5EA7", "xxxx", ""),
}


def runs(states_by_frame):
    frames = sorted(states_by_frame)
    start = 0
    for i in range(1, len(frames) + 1):
        if i == len(frames) or states_by_frame[frames[i]] != states_by_frame[frames[start]]:
            yield frames[start], frames[i - 1] + 1, states_by_frame[frames[start]]
            start = i


figure, axis = plt.subplots(figsize=(11, 8))
names = list(CLIPS)
for row, name in enumerate(names):
    clip = CLIPS[name]
    for offset, group in ((0.0, "through"), (0.36, "left")):
        for a, b, state in runs(signals[name][group]):
            colour, hatch, letter = STATE_STYLE[state]
            x0, width = (a - clip["start"]) / FPS, (b - a) / FPS
            axis.broken_barh([(x0, width)], (row + offset, 0.32), facecolors=colour, hatch=hatch, edgecolor="white", linewidth=1)
            if letter and width >= 0.7:
                axis.text(x0 + width / 2, row + offset + 0.16, letter, ha="center", va="center", fontsize=7, color="white")
axis.set_yticks([i + 0.34 for i in range(len(names))], [f"{n} ({CLIPS[n]['role'][:3]})" for n in names], fontsize=9)
axis.invert_yaxis()
axis.set_xlabel("seconds from the start of the clip")
axis.set_title("Signal state read from pixels: through (upper bar) and left (lower bar)", fontsize=11, loc="left")
axis.grid(axis="x", color="#E6E6E6", linewidth=0.8)
axis.set_axisbelow(True)
for side in ("top", "right", "left"):
    axis.spines[side].set_visible(False)
axis.tick_params(axis="y", length=0)
axis.legend(handles=[Patch(facecolor=c, hatch=h, edgecolor="white", label=s) for s, (c, h, _) in STATE_STYLE.items()], ncol=5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.07))
plt.tight_layout()
plt.show()

**How often is the reader right?** The labels give the signal state seen at frames sampled every 20 frames, leaving out the moments of a change. On a monochrome image the reader should answer `unknown`, so that is not counted as wrong.

In [11]:
def score_signals(names):
    table = Counter()
    wrong = []
    for name in names:
        clip = CLIPS[name]
        for group, intervals in ALL_LABELS[name]["signal"].items():
            for k in range(clip["start"], clip["end"], 20):
                truth = next((s for a, b, s in intervals if a <= k <= b), None)
                if truth is None:
                    continue
                read = signals[name][group][k]
                key = (clip["signals"]["method"], clip["lighting"], group)
                table[key + ("frames",)] += 1
                if read == truth:
                    table[key + ("correct",)] += 1
                elif read in ("unknown", "conflict"):
                    table[key + ("no answer",)] += 1
                else:
                    table[key + ("wrong",)] += 1
                    wrong.append((name, k, group, truth, read))
    return table, wrong


signal_table, signal_wrong = score_signals(CLIPS)
print(f"{'method':<7}{'lighting':<10}{'group':<9}{'frames':>7}{'correct':>9}{'no answer':>11}{'wrong':>7}")
for key in sorted({k[:3] for k in signal_table}):
    print(f"{key[0]:<7}{key[1]:<10}{key[2]:<9}{signal_table[key + ('frames',)]:>7}{signal_table[key + ('correct',)]:>9}{signal_table[key + ('no answer',)]:>11}{signal_table[key + ('wrong',)]:>7}")
print("\nwrong readings (clip, frame, group, labelled, read):", signal_wrong or "none")

method lighting  group     frames  correct  no answer  wrong
head   infrared  left          32        0         32      0
head   infrared  through       32        0         32      0
head   night     left         169      166          3      0
head   night     through      166      163          3      0
lamps  day       left          62        1         61      0
lamps  day       through       62       62          0      0

wrong readings (clip, frame, group, labelled, read): none


## 6. Results on the development clips

These clips shaped the rule, and their labels were made after seeing its output, so **this is not a fair test**. Recall is still far lower than in section 7, for a reason that matters: clip H is entirely monochrome, so the rule declines to judge it.

In [12]:
def as_track(track_id, rows):
    class_id = Counter(r[5] for r in rows).most_common(1)[0][0]
    return {"track_id": track_id, "class_name": COCO_VEHICLES.get(class_id, str(class_id)), "frames": [r[0] for r in rows], "boxes": [r[1:5] for r in rows]}


def run_rule(name, states=None, clip=None, rule=RULE):
    clip = clip or CLIPS[name]
    states = states or signals[name]
    records = [evaluate_track(as_track(t, rows), clip, states, LANE_ANGLE, W, H, rule) for t, rows in tracks[name].items()]
    return deduplicate([r for r in records if r], FPS, rule)


def score_clip(name, records, max_x=None):
    candidates = [r for r in records if r["verdict"] == CANDIDATE]
    return score_candidates(candidates, ALL_LABELS[name]["events"], MATCH["max_frame_difference"], max_x or MATCH["max_x_difference"])


def wilson(successes, total, z=1.96):
    # Wilson score interval: honest for small samples, unlike a normal approximation.
    if total == 0:
        return (float("nan"), float("nan"))
    p = successes / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    half = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return (max(0.0, center - half), min(1.0, center + half))


def score_table(names, title):
    print(title)
    print(f"{'clip':<6}{'crossings':>10}{'candidates':>11}{'TP':>4}{'FP':>4}{'ignored':>8}{'FN':>4}{'not eval.':>10}{'stopped':>8}{'not red':>8}{'right':>6}")
    total = Counter()
    for name in names:
        records = results[name]
        outcome = scores[name]
        verdicts = Counter(r["verdict"] for r in records)
        row = {
            "crossings": len(records),
            "candidates": verdicts[CANDIDATE],
            "TP": len(outcome["true_positives"]),
            "FP": len(outcome["false_positives"]),
            "ignored": len(outcome["ignored"]),
            "FN": len(outcome["false_negatives"]),
            "not eval.": sum(verdicts[v] for v in (AT_BORDER, OUT_OF_RANGE, UNKNOWN_SIGNAL, ENDED)),
            "stopped": verdicts[STOPPED],
            "not red": verdicts[COMPLIANT],
            "right": verdicts[RIGHT_TURN],
        }
        total.update(row)
        print(f"{name:<6}{row['crossings']:>10}{row['candidates']:>11}{row['TP']:>4}{row['FP']:>4}{row['ignored']:>8}{row['FN']:>4}{row['not eval.']:>10}{row['stopped']:>8}{row['not red']:>8}{row['right']:>6}")
    print(f"{'total':<6}{total['crossings']:>10}{total['candidates']:>11}{total['TP']:>4}{total['FP']:>4}{total['ignored']:>8}{total['FN']:>4}{total['not eval.']:>10}{total['stopped']:>8}{total['not red']:>8}{total['right']:>6}")
    tp, fp, fn = total["TP"], total["FP"], total["FN"]
    p_low, p_high = wilson(tp, tp + fp)
    r_low, r_high = wilson(tp, tp + fn)
    print(f"\nprecision {tp}/{tp + fp} = {tp / (tp + fp) if tp + fp else float('nan'):.2f}  (95% interval {p_low:.2f}-{p_high:.2f})")
    print(f"recall    {tp}/{tp + fn} = {tp / (tp + fn) if tp + fn else float('nan'):.2f}  (95% interval {r_low:.2f}-{r_high:.2f})")
    return total


def explain_outcomes(names):
    for name in names:
        outcome = scores[name]
        for candidate, event in outcome["true_positives"]:
            print(f"TP  {name} {event['id']:<5} t{candidate['track_id']:<4} {candidate['class_name']:<11} f{candidate['frame']} x={candidate['x']:<7} lane={candidate['lane']:<10} red for {candidate['state_for_frames'] / FPS:4.1f} s, entry seen: {candidate['entry_seen']}")
        for candidate, event in outcome["false_positives"]:
            print(f"FP  {name} {'-' if event is None else event['id']:<5} t{candidate['track_id']:<4} {candidate['class_name']:<11} f{candidate['frame']} x={candidate['x']:<7} lane={candidate['lane']:<10} red for {candidate['state_for_frames'] / FPS:4.1f} s, entry seen: {candidate['entry_seen']}" + ("" if event is None else f" | {event['note']}"))
        for candidate, event in outcome["ignored"]:
            print(f"IGN {name} {event['id']:<5} t{candidate['track_id']:<4} {candidate['class_name']:<11} f{candidate['frame']} x={candidate['x']:<7} | {event['note']}")
        for event in outcome["false_negatives"]:
            near = [r for r in results[name] if abs(r["frame"] - event["frame"]) <= 90 and abs(r["x"] - event["x"]) <= 250]
            reason = "; ".join(f"t{r['track_id']} f{r['frame']} x={r['x']} {r['verdict']} ({r['signal_state']})" for r in sorted(near, key=lambda r: abs(r["frame"] - event["frame"]))[:3]) or "no crossing record near it"
            print(f"FN  {name} {event['id']:<5} f{event['frame']} x={event['x']:<7} -> {reason}")


results = {name: run_rule(name) for name in CLIPS}
scores = {name: score_clip(name, results[name]) for name in CLIPS}
development_total = score_table(DEVELOPMENT, "Development clips (labels not blind)")
print()
explain_outcomes(DEVELOPMENT)

Development clips (labels not blind)
clip   crossings candidates  TP  FP ignored  FN not eval. stopped not red right
B              8          1   1   0       0   0         4       3       0     0
F              3          1   1   0       0   0         1       1       0     0
H              4          0   0   0       0   5         4       0       0     0
total         15          2   2   0       0   5         9       4       0     0

precision 2/2 = 1.00  (95% interval 0.34-1.00)
recall    2/7 = 0.29  (95% interval 0.08-0.64)

TP  B B1    t23   truck       f718 x=712.4   lane=through    red for 12.0 s, entry seen: False
TP  F F1    t2    car         f1761 x=1053.6  lane=through    red for  3.3 s, entry seen: True
FN  H H1    f2565 x=1080    -> t1 f2569 x=1077.1 not_evaluable_signal (unknown); t7 f2643 x=928.2 not_evaluable_signal (unknown)
FN  H H2    f2590 x=880     -> t1 f2569 x=1077.1 not_evaluable_signal (unknown); t7 f2643 x=928.2 not_evaluable_signal (unknown)
FN  H H3    f2590 x

## 7. Results on the evaluation clips

The main result of this notebook. The rule's logic has not changed since it was committed, before these clips were labelled.

- **TP / FP / FN:** a candidate matched to a violation, a candidate matched to nothing or to a `not_violation` label, and a labelled violation with no candidate.
- **ignored:** a candidate matched to a label marked `uncertain`, excluded from both scores.
- **not eval.:** crossings the rule declined to judge.

In [13]:
evaluation_total = score_table(EVALUATION, "Evaluation clips")
print()
explain_outcomes(EVALUATION)

Evaluation clips
clip   crossings candidates  TP  FP ignored  FN not eval. stopped not red right
A              3          1   1   0       0   0         2       0       0     0
C              2          2   2   0       0   0         0       0       0     0
D              7          1   1   0       0   0         4       2       0     0
E              1          1   1   0       0   0         0       0       0     0
G              1          1   1   0       0   0         0       0       0     0
I              1          0   0   0       0   0         1       0       0     0
J              1          1   1   0       0   0         0       0       0     0
K              2          1   1   0       0   0         0       0       1     0
L              1          1   1   0       0   0         0       0       0     0
M              1          0   0   0       0   1         1       0       0     0
N              2          2   2   0       0   0         0       0       0     0
O              1       

**How far can these numbers be trusted?** Three checks on the evaluation itself, and not on the rule:

1. **Did the matching limits do the work?** If true positives sat near the limits, a looser criterion would be inflating the score.
2. **How hard was the test?** Precision says little if nearly every crossing is a violation anyway.
3. **What does the rule add over a trivial one?** The baseline flags any crossing of the line on red, with no other condition.
4. **Is the evaluation set representative of the video?** Clip H, the only entirely monochrome clip, happens to be a development clip, so the evaluation clips leave that condition out.

In [14]:
pairs = [(abs(c["frame"] - e["frame"]), abs(c["x"] - e["x"])) for name in EVALUATION for c, e in scores[name]["true_positives"]]
d = np.array(pairs)
print("1. True positives against their labels")
print(f"   frame difference: median {np.median(d[:, 0]):.0f}, max {d[:, 0].max():.0f} (limit {MATCH['max_frame_difference']})")
print(f"   x difference:     median {np.median(d[:, 1]):.0f} px, max {d[:, 1].max():.0f} px (limit {MATCH['max_x_difference']})")

records = [r for name in EVALUATION for r in results[name]]
print("\n2. All crossing records on the evaluation clips")
print("   signal at entry:", dict(Counter(r["signal_state"] for r in records)))
print("   verdicts:       ", dict(Counter(r["verdict"] for r in records)))
print("   median tracked vehicles per clip:", int(np.median([len(tracks[n]) for n in EVALUATION])))

baseline = Counter()
baseline_false = []
for name in EVALUATION:
    outcome = score_candidates([r for r in results[name] if r["signal_state"] == "red"], ALL_LABELS[name]["events"], MATCH["max_frame_difference"], MATCH["max_x_difference"])
    baseline.update(TP=len(outcome["true_positives"]), FP=len(outcome["false_positives"]), ignored=len(outcome["ignored"]), FN=len(outcome["false_negatives"]))
    baseline_false += [(name, c["track_id"], c["frame"], c["x"], c["verdict"], e["id"] if e else None) for c, e in outcome["false_positives"]]
print("\n3. Baseline: any crossing on red")
print(f"   TP {baseline['TP']}, FP {baseline['FP']}, ignored {baseline['ignored']}, FN {baseline['FN']}: precision {baseline['TP'] / (baseline['TP'] + baseline['FP']):.2f}, recall {baseline['TP'] / (baseline['TP'] + baseline['FN']):.2f}")
print("   its false positives (clip, track, frame, x, the rule's verdict, label):")
for row in baseline_false:
    print("    ", row)

tp = development_total["TP"] + evaluation_total["TP"]
fp = development_total["FP"] + evaluation_total["FP"]
fn = development_total["FN"] + evaluation_total["FN"]
print("\n4. All 18 clips together (the labels of B, F and H are not blind)")
print(f"   precision {tp}/{tp + fp} = {tp / (tp + fp):.2f} (95% interval {wilson(tp, tp + fp)[0]:.2f}-{wilson(tp, tp + fp)[1]:.2f})")
print(f"   recall    {tp}/{tp + fn} = {tp / (tp + fn):.2f} (95% interval {wilson(tp, tp + fn)[0]:.2f}-{wilson(tp, tp + fn)[1]:.2f})")

1. True positives against their labels
   frame difference: median 2, max 6 (limit 30)
   x difference:     median 32 px, max 85 px (limit 200)

2. All crossing records on the evaluation clips
   signal at entry: {'red': 24, 'yellow': 1}
   verdicts:        {'not_evaluable_border': 6, 'candidate': 14, 'not_evaluable_ended': 2, 'stopped_at_line': 2, 'signal_not_red': 1}
   median tracked vehicles per clip: 3

3. Baseline: any crossing on red
   TP 14, FP 4, ignored 6, FN 0: precision 0.78, recall 1.00
   its false positives (clip, track, frame, x, the rule's verdict, label):
     ('A', '4', 127, 1256.3, 'not_evaluable_border', None)
     ('D', '5', 1123, 1257.0, 'not_evaluable_border', None)
     ('D', '1', 1173, 962.0, 'stopped_at_line', 'D-n1')
     ('D', '16', 1344, 328.8, 'stopped_at_line', None)

4. All 18 clips together (the labels of B, F and H are not blind)
   precision 15/15 = 1.00 (95% interval 0.80-1.00)
   recall    15/21 = 0.71 (95% interval 0.50-0.86)


**Evidence for every candidate and every missed violation.** Each row shows about 0.7 s before, the moment of entry, and about 0.8 s after. The tag gives the outcome and the signal state read at that frame; the box is the tracked vehicle.

In [15]:
def evidence_row(name, frame, tag, track_id=None, group="through"):
    clip = CLIPS[name]
    rows = {r[0]: r for r in tracks[name].get(track_id, [])} if track_id is not None else {}
    tiles = []
    for k in (max(clip["start"], frame - 20), frame, min(clip["end"] - 1, frame + 25)):
        image = read_frame_at(VIDEO, k)["image"]
        cv2.polylines(image, [np.array(clip["stop_line"], np.int32)], False, (0, 255, 0), 2, cv2.LINE_AA)
        if k in rows:
            cv2.rectangle(image, tuple(int(v) for v in rows[k][1:3]), tuple(int(v) for v in rows[k][3:5]), (0, 255, 255), 3)
        tile = cv2.resize(image, (400, 225), interpolation=cv2.INTER_AREA)
        cv2.rectangle(tile, (0, 200), (400, 225), (0, 0, 0), -1)
        cv2.putText(tile, f"{tag} {name} f{k} signal: {signals[name][group].get(k, '?')}", (6, 218), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        tiles.append(tile)
    return np.hstack(tiles)


def evidence_rows(names):
    rows = []
    for name in names:
        outcome = scores[name]
        for tag, pairs in (("TP", outcome["true_positives"]), ("FP", outcome["false_positives"]), ("IGN", outcome["ignored"])):
            for candidate, _ in pairs:
                rows.append(evidence_row(name, candidate["frame"], tag, candidate["track_id"], candidate["signal_group"]))
        for event in outcome["false_negatives"]:
            rows.append(evidence_row(name, event["frame"], "FN"))
    return rows


rows = evidence_rows(EVALUATION)
for i in range(0, len(rows), 8):
    show(np.vstack(rows[i:i + 8]))

## 8. Validation

### 8.1 Does the signal really control the result?

A candidate must depend on the signal. If the state is replaced, the outcome should follow. This is a check of the logic, on the evaluation clips.

In [16]:
def with_state(name, value):
    return {g: {k: value for k in v} for g, v in signals[name].items()}


counterfactual = {}
for value in ("green", "yellow", "unknown", "conflict", "red"):
    counterfactual[value] = sum(r["verdict"] == CANDIDATE for name in EVALUATION for r in run_rule(name, states=with_state(name, value)))
    print(f"signal forced to {value:<9} -> {counterfactual[value]} candidates")
print(f"signal as read            -> {sum(r['verdict'] == CANDIDATE for name in EVALUATION for r in results[name])} candidates")
assert counterfactual["green"] == counterfactual["yellow"] == counterfactual["unknown"] == counterfactual["conflict"] == 0

signal forced to green     -> 0 candidates
signal forced to yellow    -> 0 candidates
signal forced to unknown   -> 0 candidates
signal forced to conflict  -> 0 candidates
signal forced to red       -> 15 candidates
signal as read            -> 14 candidates


### 8.2 How sensitive is the result to its settings?

Each row changes one setting and rescores the evaluation clips. **This is not a search for better settings**: choosing them here would turn the evaluation clips into development clips. It shows how much the result depends on values that could easily have been different, such as a stop line that is 10 px off.

In [17]:
def shifted_line(clip, dy):
    return {**clip, "stop_line": [[x, y + dy] for x, y in clip["stop_line"]]}


def shifted_states(name, frames):
    return {g: {k + frames: s for k, s in v.items()} for g, v in signals[name].items()}


def score_variant(clip_change=None, rule=RULE, delay=0, max_x=None):
    total = Counter()
    changes = []
    for name in EVALUATION:
        clip = clip_change(CLIPS[name]) if clip_change else CLIPS[name]
        states = shifted_states(name, delay) if delay else signals[name]
        outcome = score_clip(name, run_rule(name, states=states, clip=clip, rule=rule), max_x=max_x)
        total.update(TP=len(outcome["true_positives"]), FP=len(outcome["false_positives"]), ignored=len(outcome["ignored"]), FN=len(outcome["false_negatives"]))
        changes += [f"FP {name} t{c['track_id']} f{c['frame']} x={c['x']}" + (f" ({e['id']})" if e else "") for c, e in outcome["false_positives"]]
        changes += [f"FN {name} {e['id']}" for e in outcome["false_negatives"]]
    return total, changes


variants = {
    "as frozen": {},
    "stop line 20 px up": {"clip_change": lambda c: shifted_line(c, -20)},
    "stop line 10 px up": {"clip_change": lambda c: shifted_line(c, -10)},
    "stop line 10 px down": {"clip_change": lambda c: shifted_line(c, 10)},
    "stop line 20 px down": {"clip_change": lambda c: shifted_line(c, 20)},
    "entry offset 0.20": {"rule": {**RULE, "entry_offset": 0.20}},
    "entry offset 0.40": {"rule": {**RULE, "entry_offset": 0.40}},
    "pass margin 0.25": {"rule": {**RULE, "pass_margin": 0.25}},
    "pass margin 0.75": {"rule": {**RULE, "pass_margin": 0.75}},
    "signal read 0.5 s early": {"delay": -15},
    "signal read 0.5 s late": {"delay": 15},
}
print(f"{'variant':<26}{'TP':>4}{'FP':>4}{'ignored':>9}{'FN':>4}{'precision':>11}{'recall':>8}")
def print_variants(max_x=None):
    print(f"{'variant':<26}{'TP':>4}{'FP':>4}{'ignored':>9}{'FN':>4}{'precision':>11}{'recall':>8}  errors")
    for title, change in variants.items():
        t, errors = score_variant(**change, max_x=max_x)
        precision = t["TP"] / (t["TP"] + t["FP"]) if t["TP"] + t["FP"] else float("nan")
        recall = t["TP"] / (t["TP"] + t["FN"]) if t["TP"] + t["FN"] else float("nan")
        print(f"{title:<26}{t['TP']:>4}{t['FP']:>4}{t['ignored']:>9}{t['FN']:>4}{precision:>11.2f}{recall:>8.2f}  {'; '.join(errors)}")


print(f"With the committed matching limit of {MATCH['max_x_difference']} px:")
print_variants()
print("\nWith the x limit tightened to 150 px (true positives differ from their labels by at most 85 px):")
print_variants(max_x=150)

variant                     TP  FP  ignored  FN  precision  recall
With the committed matching limit of 200 px:
variant                     TP  FP  ignored  FN  precision  recall  errors
as frozen                   13   0        1   1       1.00    0.93  FN M M1


stop line 20 px up          13   1        1   1       0.93    0.93  FP D t12 f1173 x=793.3 (D-n1); FN D D1
stop line 10 px up          12   1        1   2       0.92    0.86  FP D t12 f1168 x=766.6 (D-n1); FN D D1; FN M M1
stop line 10 px down        13   0        0   1       1.00    0.93  FN M M1
stop line 20 px down        13   0        0   1       1.00    0.93  FN M M1


entry offset 0.20           12   1        1   2       0.92    0.86  FP D t12 f1168 x=766.6 (D-n1); FN D D1; FN M M1
entry offset 0.40           13   0        1   1       1.00    0.93  FN M M1


pass margin 0.25            13   0        1   1       1.00    0.93  FN M M1
pass margin 0.75            11   0        1   3       1.00    0.79  FN C C1; FN C C2; FN M M1
signal read 0.5 s early     13   0        1   1       1.00    0.93  FN M M1
signal read 0.5 s late      11   0        0   3       1.00    0.79  FN C C1; FN M M1; FN N N2

With the x limit tightened to 150 px (true positives differ from their labels by at most 85 px):
variant                     TP  FP  ignored  FN  precision  recall  errors
as frozen                   13   0        1   1       1.00    0.93  FN M M1


stop line 20 px up          13   1        1   1       0.93    0.93  FP G t1 f2038 x=615.4; FN G G1


stop line 10 px up          13   0        1   1       1.00    0.93  FN M M1
stop line 10 px down        13   0        0   1       1.00    0.93  FN M M1
stop line 20 px down        13   0        0   1       1.00    0.93  FN M M1
entry offset 0.20           12   1        1   2       0.92    0.86  FP G t1 f2038 x=615.4; FN G G1; FN M M1
entry offset 0.40           13   0        1   1       1.00    0.93  FN M M1


pass margin 0.25            13   0        1   1       1.00    0.93  FN M M1
pass margin 0.75            11   0        1   3       1.00    0.79  FN C C1; FN C C2; FN M M1


signal read 0.5 s early     13   0        1   1       1.00    0.93  FN M M1
signal read 0.5 s late      11   0        0   3       1.00    0.79  FN C C1; FN M M1; FN N N2


### 8.3 Logic test on a video with a synthetic signal

**This is not evidence about real violations.** In `traffic_video_modified.mp4` the signal is a graphic added to the footage: the vehicles never saw it, so a "red-light crossing" here means nothing about any driver. Nothing from this section enters the candidates file.

It is still useful, because the true signal state is known exactly in every frame, and the camera moves and the traffic differs. It tests that the rule's logic behaves as designed outside the footage it was built on.

Two adaptations, both made through the rule's parameters and not by changing its code:

- **Camera motion:** tracks are mapped to the coordinates of a reference frame by a similarity transform, as in Notebook 04.
- **Direction:** traffic approaches the camera here, while the rule assumes traffic moving away. So coordinates are mirrored vertically, and the entry offset is 0, because the bottom of an approaching vehicle's box is already its front.

The stop line in reference coordinates was declared by eye. The turn logic is not meaningful under mirroring, so this test does not cover it.

In [18]:
OVERLAY_BOX = (1750, 29, 1871, 301)
OVERLAY_LAMPS = {"red": (1810, 79), "yellow": (1810, 160), "green": (1810, 239)}
OVERLAY_WATERMARK = (500, 410, 1420, 670)
OVERLAY_LINE = [[1045.0, 852.0], [1864.0, 888.0]]
OVERLAY_PATH = OUTPUT_DIR / "overlay_run.json"
MIRROR = 2000.0


def overlay_mask(shape, detections):
    mask = np.full(shape, 255, np.uint8)
    for x1, y1, x2, y2, *_ in detections:
        mask[max(0, int(y1) - 12):int(y2) + 12, max(0, int(x1) - 12):int(x2) + 12] = 0
    x1, y1, x2, y2 = OVERLAY_BOX
    mask[y1 - 8:y2 + 8, x1 - 8:x2 + 8] = 0
    x1, y1, x2, y2 = OVERLAY_WATERMARK
    mask[y1:y2, x1:x2] = 0
    return mask


def run_overlay():
    # Detection, tracking, signal reading and registration of the overlay video.
    model, new_tracker = load_detector_and_tracker_factory()
    tracker = new_tracker()
    raw_states, detections_per_frame, overlay_tracks = [], [], {}
    for record in iter_frames(OVERLAY_VIDEO):
        image, k = record["image"], record["frame_index"]
        raw_states.append(read_lamp_state(image, OVERLAY_LAMPS, half=12))
        detections = detect(model, image)
        detections_per_frame.append(detections)
        output = tracker.update(detections, image)
        for row in output if len(output) else []:
            overlay_tracks.setdefault(str(int(row[4])), []).append([k, *[round(float(v), 1) for v in row[:4]], int(detections[int(row[7]), 5]), round(float(row[5]), 3)])
    orb, matcher = cv2.ORB_create(6000), cv2.BFMatcher(cv2.NORM_HAMMING)
    reference = len(raw_states) // 2
    gray = cv2.cvtColor(read_frame_at(OVERLAY_VIDEO, reference)["image"], cv2.COLOR_BGR2GRAY)
    ref_points, ref_descriptors = orb.detectAndCompute(gray, overlay_mask(gray.shape, detections_per_frame[reference]))
    params = []
    for record in iter_frames(OVERLAY_VIDEO):
        gray = cv2.cvtColor(record["image"], cv2.COLOR_BGR2GRAY)
        points, descriptors = orb.detectAndCompute(gray, overlay_mask(gray.shape, detections_per_frame[record["frame_index"]]))
        good = [p[0] for p in matcher.knnMatch(descriptors, ref_descriptors, k=2) if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
        matrix, _ = cv2.estimateAffinePartial2D(
            np.float32([points[m.queryIdx].pt for m in good]), np.float32([ref_points[m.trainIdx].pt for m in good]),
            method=cv2.RANSAC, ransacReprojThreshold=2.0, maxIters=5000,
        )
        params.append([float(matrix[0, 0]), float(matrix[1, 0]), float(matrix[0, 2]), float(matrix[1, 2])])
    return {"states": raw_states, "tracks": overlay_tracks, "params": params}


overlay = None
if not OVERLAY_VIDEO.exists():
    print("Overlay video not found; this section is skipped.")
elif OVERLAY_PATH.exists():
    overlay = json.loads(OVERLAY_PATH.read_text())
else:
    overlay = run_overlay()
    OVERLAY_PATH.write_text(json.dumps(overlay))

if overlay:
    overlay_meta = read_video_metadata(OVERLAY_VIDEO)
    overlay_fps, overlay_h = overlay_meta["fps"], overlay_meta["height"]
    overlay_states = dict(enumerate(smooth_states(overlay["states"])))
    print(f"{len(overlay_states)} frames, {overlay_fps:.2f} FPS, {len(overlay['tracks'])} tracks")
    print("signal phases read from the overlay (state, from s, to s):")
    for a, b, state in runs(overlay_states):
        print(f"  {state:<8}{a / overlay_fps:>7.2f}{b / overlay_fps:>7.2f}")

715 frames, 26.33 FPS, 44 tracks
signal phases read from the overlay (state, from s, to s):
  green      0.00   8.01
  yellow     8.01  11.01
  red       11.01  15.00
  green     15.00  20.01
  yellow    20.01  22.03
  red       22.03  27.15


In [19]:
def mirrored_track(track_id, rows, params):
    # The contact point of each box in mirrored reference coordinates, with the box's own size.
    boxes = []
    for k, x1, y1, x2, y2, *_ in rows:
        rx, ry = frame_to_reference(params[k], [[(x1 + x2) / 2, y2]])[0]
        bottom = MIRROR - ry
        boxes.append([rx - (x2 - x1) / 2, bottom - (y2 - y1), rx + (x2 - x1) / 2, bottom])
    class_id = Counter(r[5] for r in rows).most_common(1)[0][0]
    return {"track_id": track_id, "class_name": COCO_VEHICLES.get(class_id, str(class_id)), "frames": [r[0] for r in rows], "boxes": boxes}


if overlay:
    line = [[x, MIRROR - y] for x, y in OVERLAY_LINE]
    logic_rule = {**RULE, "entry_offset": 0.0}
    logic_clip = {"start": 0, "end": len(overlay_states), "stop_line": line, "lane_dividers": [line[0][0], line[-1][0]], "evaluable_x_max": 10**6}
    logic_states = {"through": overlay_states, "left": overlay_states}

    # The direction of traffic comes from the tracks that cross the line, not from the rule.
    headings = []
    for track_id, rows in overlay["tracks"].items():
        track = mirrored_track(track_id, rows, overlay["params"])
        if len(rows) >= 30:
            b = np.array(track["boxes"])
            margin = b[:, 3] - line_y(line, (b[:, 0] + b[:, 2]) / 2)
            if margin[0] > 0 and margin[-1] < 0:
                d = b[-1, [0, 3]] - b[0, [0, 3]]
                headings.append(np.arctan2(d[1], d[0]))
    logic_angle = float(np.degrees(np.angle(np.mean(np.exp(1j * np.array(headings))))))
    print(f"{len(headings)} tracks cross the line; median direction in mirrored coordinates: {logic_angle:.0f} degrees")

    logic_records = []
    for track_id, rows in overlay["tracks"].items():
        record = evaluate_track(mirrored_track(track_id, rows, overlay["params"]), logic_clip, logic_states, logic_angle, 10**6, 10**6, logic_rule)
        if record:
            real = next(r for r in rows if r[0] == record["frame"])
            if real[4] >= overlay_h - 3:
                record["verdict"] = AT_BORDER
            logic_records.append(record)
    logic_records.sort(key=lambda r: r["frame"])
    print(f"\n{'frame':>6}{'time s':>8}  {'signal':<9}{'track':>6}  {'class':<11}{'x':>7}  {'passed':<7}{'verdict'}")
    for r in logic_records:
        print(f"{r['frame']:>6}{r['frame'] / overlay_fps:>8.2f}  {r['signal_state']:<9}{r['track_id']:>6}  {r['class_name']:<11}{r['x']:>7.0f}  {r['passed_line']!s:<7}{r['verdict']}")
    print("\nverdicts:", dict(Counter(r["verdict"] for r in logic_records)))
    print("By signal state at entry:", dict(Counter((r["signal_state"], r["verdict"]) for r in logic_records)))
    assert all(r["signal_state"] == "red" for r in logic_records if r["verdict"] == CANDIDATE)

7 tracks cross the line; median direction in mirrored coordinates: -169 degrees

 frame  time s  signal    track  class            x  passed verdict
    50    1.90  green         6  car           1497  False  signal_not_red
   102    3.87  green         9  car           1467  False  signal_not_red
   154    5.85  green         3  car           1480  False  signal_not_red
   238    9.04  yellow        7  car           1560  False  signal_not_red
   317   12.04  red          23  car           1482  True   candidate
   624   23.70  red          43  truck         1036  False  stopped_at_line

verdicts: {'signal_not_red': 4, 'candidate': 1, 'stopped_at_line': 1}
By signal state at entry: {('green', 'signal_not_red'): 3, ('yellow', 'signal_not_red'): 1, ('red', 'candidate'): 1, ('red', 'stopped_at_line'): 1}


In [20]:
if overlay:
    tiles = []
    for r in [r for r in logic_records if r["verdict"] == CANDIDATE][:4]:
        rows = {row[0]: row for row in overlay["tracks"][r["track_id"]]}
        for k in (r["frame"] - 12, r["frame"], min(len(overlay_states) - 1, r["frame"] + 12)):
            image = read_frame_at(OVERLAY_VIDEO, k)["image"]
            cv2.polylines(image, [np.round(reference_to_frame(overlay["params"][k], OVERLAY_LINE)).astype(np.int32)], False, (0, 255, 0), 3, cv2.LINE_AA)
            if k in rows:
                cv2.rectangle(image, tuple(int(v) for v in rows[k][1:3]), tuple(int(v) for v in rows[k][3:5]), (0, 255, 255), 4)
            tile = cv2.resize(image, (400, round(image.shape[0] * 400 / image.shape[1])), interpolation=cv2.INTER_AREA)
            cv2.putText(tile, f"SYNTHETIC SIGNAL f{k}: {overlay_states[k]}", (6, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)
            tiles.append(tile)
    if tiles:
        show(grid(tiles, 3), "Logic test only: the signal is a graphic the drivers never saw. Not a violation.")

## 9. Candidate events

Every candidate from the real clips becomes an event with status `pending`, ready for human review in a later notebook. Development and evaluation clips are marked. Each crossing record is kept too, so a reviewer can see what the rule declined to judge.

In [21]:
events = [
    {**build_event(r, name, VIDEO_ID, FPS, CLIPS[name]["start"], rule=RULE), "clip_role": CLIPS[name]["role"]}
    for name in CLIPS
    for r in results[name]
    if r["verdict"] == CANDIDATE
]
payload = {
    "video_id": VIDEO_ID,
    "rule": RULE,
    "tracking_session": "yolo26m.ocsort-byte",
    "signal_source": "camera",
    "events": events,
    "crossings": results,
    "scores": {
        "development": {k: development_total[k] for k in ("TP", "FP", "ignored", "FN")},
        "evaluation": {k: evaluation_total[k] for k in ("TP", "FP", "ignored", "FN")},
        "labels_reviewed_by_a_person": labels["annotator"]["reviewed_by_a_person"],
    },
}
CANDIDATES_PATH = OUTPUT_DIR / f"{VIDEO_ID}.red_light.candidates.json"
CANDIDATES_PATH.write_text(json.dumps(payload, indent=1))

# Data contract.
assert len({e["event_id"] for e in events}) == len(events)
assert all(e["review_status"] == "pending" and e["license_plate"] is None and e["signal_source"] == "camera" for e in events)
assert "traffic_video_modified" not in CANDIDATES_PATH.read_text()
print(f"{len(events)} candidate events written to {CANDIDATES_PATH.relative_to(PROJECT_ROOT)}")
print(json.dumps({k: events[0][k] for k in ("event_id", "clip", "trigger_time_s", "rule", "why", "license_plate", "review_status")}, indent=1) if events else "no events")

16 candidate events written to outputs/05b_red_light_violation/rlr_occasions.red_light.candidates.json
{
 "event_id": "evt-rlr_occasions-A-red_light-f00153-t12",
 "clip": "A",
 "trigger_time_s": 5.1,
 "rule": {
  "name": "red_light",
  "version": 1
 },
 "why": "Entered in lane 'through' while the through signal was red (2.5 s after it was first seen in that state), and continued past the stop line.",
 "license_plate": null,
 "review_status": "pending"
}


## 10. Clips of the candidate events

The output a user actually needs from this step: **for each candidate, a short video of that case**, handed on to plate reading in Notebook 06.

- **Original pixels.** A clip is cut in time only: nothing is drawn on it, cropped from it or blurred, and it is compressed close to losslessly.
- **It starts early.** A vehicle seen from behind is nearest the camera, with its plate largest, *before* it reaches the line. So a clip reaches back to the start of the vehicle's track, up to 10 s, and at least 3 s. It ends 2 s after entry, and never crosses a cut in the source.
- **It says where the vehicle is.** `manifest.json` gives the vehicle's box in each frame, so the next step knows which of several vehicles to read.

Candidates are still `pending`: a clip is material for review, not proof of a violation.

In [22]:
CLIPS_DIR = OUTPUT_DIR / "violation_clips"
entries = [
    cut_event_clip(VIDEO, event, tracks[event["clip"]][event["track_id"]], CLIPS[event["clip"]]["start"], CLIPS[event["clip"]]["end"], CLIPS_DIR)
    for event in events
]
clip_manifest = {
    "schema_version": 1,
    "video_id": VIDEO_ID,
    "source_video": str(VIDEO.relative_to(PROJECT_ROOT)),
    "settings": CLIP_SETTINGS,
    "clips": entries,
}
(CLIPS_DIR / "manifest.json").write_text(json.dumps(clip_manifest, indent=1))
print(f"{len(entries)} clips and manifest.json in {CLIPS_DIR.relative_to(PROJECT_ROOT)}, {sum(p.stat().st_size for p in CLIPS_DIR.glob('*.mp4')) / 1e6:.1f} MB")

16 clips and manifest.json in outputs/05b_red_light_violation/violation_clips, 39.2 MB


**Checking the clips.** Each is decoded again and compared with the source at the moment of entry: the frame count must be exact, the image must match (PSNR, in dB), and it must be as sharp as the source (ratio of Laplacian variance). The tallest box shows how large the vehicle gets, which limits how readable its plate can be.

In [23]:
def decode_clip(path):
    capture = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        frames.append(frame)
    capture.release()
    return frames


def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 99.0 if mse == 0 else 10 * np.log10(255**2 / mse)


def sharpness(image):
    return cv2.Laplacian(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()


print(f"{'event':<44}{'seconds':>8}{'frames':>8}{'with box':>9}{'tallest box':>12}{'PSNR':>7}{'sharpness':>10}")
crops = []
for entry in entries:
    frames = decode_clip(CLIPS_DIR / entry["clip_file"])
    assert len(frames) == entry["frame_count"], entry["event_id"]
    source = read_frame_at(VIDEO, entry["source_start_frame"] + entry["trigger_frame_in_clip"])["image"]
    at_entry = frames[entry["trigger_frame_in_clip"]]
    tallest = max(entry["vehicle_boxes"], key=lambda b: b["box_xyxy"][3] - b["box_xyxy"][1])
    height = tallest["box_xyxy"][3] - tallest["box_xyxy"][1]
    quality = psnr(at_entry, source)
    assert quality >= 30, entry["event_id"]
    print(f"{entry['event_id']:<44}{entry['frame_count'] / entry['fps']:>8.1f}{entry['frame_count']:>8}{len(entry['vehicle_boxes']):>9}{height:>10.0f}px{quality:>7.1f}{sharpness(at_entry) / sharpness(source):>10.2f}")
    x1, y1, x2, y2 = [int(max(0, v)) for v in tallest["box_xyxy"]]
    crop = frames[tallest["frame_in_clip"]][y1:y2, x1:x2]
    scale = 240 / max(crop.shape[:2])
    crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    tile = np.zeros((262, 240, 3), np.uint8)
    tile[22:22 + crop.shape[0], :crop.shape[1]] = crop
    cv2.putText(tile, entry["event_id"].split("-red_light-")[0].split("-")[-1] + " " + entry["event_id"].split("-")[-1] + f" {height:.0f}px", (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    crops.append(tile)
show(grid(crops, 5), "Largest view of each vehicle in its clip, scaled up (local copy only)")

event                                        seconds  frames with box tallest box   PSNR sharpness


evt-rlr_occasions-A-red_light-f00153-t12         4.5     135       92       195px   43.4      1.00


evt-rlr_occasions-B-red_light-f00718-t23         6.3     188      131       152px   43.2      1.00


evt-rlr_occasions-C-red_light-f01027-t1          5.0     151       69       223px   42.7      1.00


evt-rlr_occasions-C-red_light-f01059-t2          5.0     151       71       186px   42.4      1.01


evt-rlr_occasions-D-red_light-f01166-t12         3.5     105       83       147px   42.5      1.00


... (7 outputs left out)

evt-rlr_occasions-N-red_light-f04282-t1          5.0     151       91       206px   43.0      1.01


evt-rlr_occasions-O-red_light-f04600-t1          5.0     151       80       179px   42.6      1.00


evt-rlr_occasions-Q-red_light-f04957-t2          5.0     151       78       150px   42.5      1.00


evt-rlr_occasions-R-red_light-f05100-t4          2.2      67       62       115px   43.7      0.99


**The same output for the synthetic-signal video.** This shows that the step works on another video, with another size and frame rate. It stays in its own folder and is marked: the signal there is a graphic, so these are **not violations**.

In [24]:
if overlay:
    logic_events = [build_event(r, "logic_test", OVERLAY_VIDEO.stem, overlay_fps, 0, signal_source="synthetic_overlay", rule=RULE) for r in logic_records if r["verdict"] == CANDIDATE]
    logic_dir = OUTPUT_DIR / "logic_test_clips"
    logic_entries = [cut_event_clip(OVERLAY_VIDEO, e, overlay["tracks"][e["track_id"]], 0, len(overlay_states), logic_dir) for e in logic_events]
    (logic_dir / "manifest.json").write_text(json.dumps({
        "schema_version": 1,
        "video_id": OVERLAY_VIDEO.stem,
        "source_video": str(OVERLAY_VIDEO.relative_to(PROJECT_ROOT)),
        "warning": "The signal in this video is a graphic added to the footage. These clips are a logic test, not violations.",
        "settings": CLIP_SETTINGS,
        "clips": logic_entries,
    }, indent=1))
    for entry in logic_entries:
        frames = decode_clip(logic_dir / entry["clip_file"])
        assert len(frames) == entry["frame_count"] and entry["signal_source"] == "synthetic_overlay"
        print(f"{entry['clip_file']}: {entry['width']}x{entry['height']}, {entry['fps']:.2f} FPS, {entry['frame_count']} frames, {len(entry['vehicle_boxes'])} with a box")
    assert not any(e["signal_source"] != "camera" for e in entries)

evt-traffic_video_modified-logic_test-red_light-f00317-t23.mp4: 1920x1012, 26.33 FPS, 133 frames, 108 with a box


## 11. Material for a person to check the labels

Every score above rests on labels no person has checked. `review/` makes that quick: one annotated clip per case, showing the stop line, the signal state the rule read, and the vehicle's box when the tracker has it in that frame. Near the labelled moment, a triangle on the line marks a vehicle with no box. A clip with no box at all does not always mean the vehicle was not detected: the box is drawn only for a track that has a crossing record. The sheet `review/review_sheet.csv` lists the cases in order of importance, and leaves two columns for the reviewer.

**What to check for each case.** Did the front of the vehicle cross the line while the signal of its lane was red, and did the vehicle continue into the junction?

In [25]:
REVIEW_DIR = OUTPUT_DIR / "review"
REVIEW_SETTINGS = {**CLIP_SETTINGS, "pre_seconds": 3.0, "max_pre_seconds": 4.0, "post_seconds": 3.0}
ORDER = {"violation": 0, "not_violation": 1, "uncertain": 2}


def labelled_crossings(name):
    # What the rule did about each labelled vehicle: its matched candidate, else its own crossing record.
    chosen = {event["id"]: candidate for key in ("true_positives", "false_positives", "ignored") for candidate, event in scores[name][key] if event is not None}
    taken = [r["track_id"] for r in chosen.values()]
    others = [e for e in ALL_LABELS[name]["events"] if e["id"] not in chosen]
    free = [r for r in results[name] if r["track_id"] not in taken]
    return {**chosen, **match_labels_to_crossings(others, free, MATCH["max_frame_difference"], MATCH["max_x_difference"])}


def review_drawer(name, case_id, event, record):
    clip = CLIPS[name]
    group = record["signal_group"] if record else ("left" if lane_of(clip, event["x"]) == "left_turn" else "through")
    rows = {r[0]: r for r in tracks[name][record["track_id"]]} if record else {}
    points = np.array(clip["stop_line"], np.int32)

    def draw(image, k):
        cv2.polylines(image, [points], False, (0, 255, 0), 2, cv2.LINE_AA)
        if k in rows:
            cv2.rectangle(image, tuple(int(v) for v in rows[k][1:3]), tuple(int(v) for v in rows[k][3:5]), (0, 255, 255), 2)
        elif abs(k - event["frame"]) <= 15:
            x, y = int(event["x"]), int(line_y(clip["stop_line"], event["x"]))
            cv2.fillPoly(image, [np.array([[x, y + 4], [x - 12, y + 26], [x + 12, y + 26]], np.int32)], (0, 255, 255))
        cv2.rectangle(image, (0, 0), (470, 32), (0, 0, 0), -1)
        cv2.putText(image, f"{case_id}  f{k}  signal read: {signals[name][group].get(k, '?')}", (8, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

    return draw


clip_of_event = {e["event_id"]: e for e in entries}
cases = []
for name, clip in CLIPS.items():
    crossings = labelled_crossings(name)
    for event in ALL_LABELS[name]["events"]:
        record = crossings.get(event["id"])
        if event["label"] == "uncertain" and not (record and record["verdict"] == CANDIDATE):
            continue
        cases.append((ORDER[event["label"]], name, event, record))
cases.sort(key=lambda c: (c[0], c[1], c[2]["frame"]))

rows_out = []
for order, (_, name, event, record) in enumerate(cases, start=1):
    clip = CLIPS[name]
    track_start = int(tracks[name][record["track_id"]][0][0]) if record else event["frame"]
    start, end = clip_frame_range(event["frame"], track_start, FPS, clip["start"], clip["end"], REVIEW_SETTINGS)
    file_name = f"{order:02d}_{event['id']}.mp4"
    write_clip(VIDEO, start, end, REVIEW_DIR / file_name, draw=review_drawer(name, event["id"], event, record))
    event_clip = next((e["clip_file"] for e in entries if record and record["verdict"] == CANDIDATE and e["event_id"].endswith(f"-t{record['track_id']}") and f"-{name}-" in e["event_id"]), "")
    rows_out.append([order, event["id"], file_name, clip["role"], event["label"], record["verdict"] if record else "no crossing record", event_clip, event["note"], "", ""])

with (REVIEW_DIR / "review_sheet.csv").open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["order", "case_id", "review_clip", "clip_role", "ai_label", "rule_result", "violation_clip", "ai_note", "your_verdict", "your_note"])
    writer.writerows(rows_out)
print(f"{len(rows_out)} review clips and review_sheet.csv in {REVIEW_DIR.relative_to(PROJECT_ROOT)}, {sum(p.stat().st_size for p in REVIEW_DIR.glob('*.mp4')) / 1e6:.1f} MB")
print(Counter(r[4] for r in rows_out))
print(f"\n{'#':>3} {'case':<7}{'AI label':<15}rule result")
for r in rows_out:
    print(f"{r[0]:>3} {r[1]:<7}{r[4]:<15}{r[5]}")

30 review clips and review_sheet.csv in outputs/05b_red_light_violation/review, 79.7 MB
Counter({'violation': 21, 'not_violation': 8, 'uncertain': 1})

  # case   AI label       rule result
  1 A1     violation      candidate
  2 B1     violation      candidate
  3 C1     violation      candidate
  4 C2     violation      candidate
  5 D1     violation      candidate
  6 E1     violation      candidate
  7 F1     violation      candidate
  8 G1     violation      candidate
  9 H1     violation      not_evaluable_signal
 10 H2     violation      not_evaluable_signal
 11 H3     violation      not_evaluable_signal
 12 H4     violation      not_evaluable_signal
 13 H5     violation      no crossing record
 14 J1     violation      candidate
 15 K1     violation      candidate
 16 L1     violation      candidate
 17 M1     violation      not_evaluable_ended
 18 N1     violation      candidate
 19 N2     violation      candidate
 20 O1     violation      candidate
 21 Q1     violation      c

## 12. From any video to its results

The steps of this notebook are now small functions in `src/traffic_violation_detection/`, and one function that joins them:

```python
from traffic_violation_detection.red_light import detect_red_light_video

result = detect_red_light_video("path/to/video.mp4", "path/to/output_folder", "path/to/scene.json")
```

It needs a third input, the **scene configuration**: the stop line and signal positions cannot be read from the video, so a new camera needs one first. Only the frame ranges of its clips are analysed and saved. It writes:

- `<video name>_output.mp4`: those frames, showing the stop line, the signal state and each vehicle's verdict;
- `<video name>_violation_clips/`: **one clip per candidate**, with original pixels, and `manifest.json`, the input to plate reading;
- `<video name>_events.json`: the candidates, all `pending`, and every crossing record.

| Function | Step |
|---|---|
| `track_frames`, `add_to_tracks` (`tracking.py`) | Read, detect and track a clip |
| `read_frame_signals`, `smooth_states` | Read each signal group in each frame |
| `track_and_read_clip` | Join those two into one pass over a clip |
| `evaluate_track`, `deduplicate`, `evaluate_clip` | The rule, on every track of a clip |
| `build_event` | Turn a candidate into a `pending` event |
| `draw_clip_frame`, `draw_banner`, `write_result_video` | Draw the result on each frame and save the video |
| `write_event_clips` (`evidence.py`) | Cut one clip per candidate |
| `detect_red_light_video` | **Joins them all** |

**Checking it.** Run on this video, it should reproduce the candidates and the scores above. The rule itself has not changed.

In [26]:
from traffic_violation_detection.red_light import detect_red_light_video

FUNCTION_DIR = OUTPUT_DIR / "function_output"
function_result = detect_red_light_video(VIDEO, FUNCTION_DIR, PROJECT_ROOT / "configs" / "scenes" / "rlr_occasions.json", rule=RULE)

key = lambda e: (e["clip"], e["track_id"], e["trigger_frame"])
function_keys, notebook_keys = {key(e) for e in function_result["events"]}, {key(e) for e in events}
print(f"candidate events: {len(function_keys)} from the function, {len(notebook_keys)} in the notebook, {len(function_keys & notebook_keys)} identical")
print("only from the function:", sorted(function_keys - notebook_keys) or "none")
print("only in the notebook:  ", sorted(notebook_keys - function_keys) or "none")

function_total = Counter()
for name in EVALUATION:
    candidates = [r for r in function_result["crossings"][name] if r["verdict"] == CANDIDATE]
    outcome = score_candidates(candidates, ALL_LABELS[name]["events"], MATCH["max_frame_difference"], MATCH["max_x_difference"])
    function_total.update(TP=len(outcome["true_positives"]), FP=len(outcome["false_positives"]), ignored=len(outcome["ignored"]), FN=len(outcome["false_negatives"]))
print("evaluation clips, function:", dict(function_total), "| notebook:", {k: evaluation_total[k] for k in ("TP", "FP", "ignored", "FN")})

expected_frames = sum(c["end"] - c["start"] for c in CLIPS.values())
output_frames = sum(1 for _ in iter_frames(function_result["output_video"]))
manifest = json.loads(function_result["clips_manifest"].read_text())
print(f"\n{function_result['output_video'].relative_to(PROJECT_ROOT)}: {output_frames} frames (clips hold {expected_frames}), {function_result['output_video'].stat().st_size / 1e6:.1f} MB")
print(f"clips: {len(manifest['clips'])}, all pending: {all(c['review_status'] == 'pending' for c in manifest['clips'])}")
assert output_frames == expected_frames and len(manifest["clips"]) == len(function_result["events"])
assert {k: function_total[k] for k in ("TP", "FP", "FN")} == {k: evaluation_total[k] for k in ("TP", "FP", "FN")}

candidate events: 16 from the function, 16 in the notebook, 16 identical
only from the function: none
only in the notebook:   none
evaluation clips, function: {'TP': 13, 'FP': 0, 'ignored': 1, 'FN': 1} | notebook: {'TP': 13, 'FP': 0, 'ignored': 1, 'FN': 1}



outputs/05b_red_light_violation/function_output/rlr_occasions_output.mp4: 5150 frames (clips hold 5150), 40.3 MB
clips: 16, all pending: True



## 13. Rule version 2: when a track breaks

Everything above is version 1, and its scores stand. Version 2 handles a vehicle whose track breaks, the cause of the miss M1. It was shaped **after seeing these clips**, so its scores here are not a fair test.

**Why not simply join any two tracks?** Joining two vehicles could give one of them the other's plate, which is the worst mistake this system can make. Measurements showed that a general join cannot be trusted here: a pair known to be one vehicle had a colour distance of 0.87, while pairs of different vehicles reached 0.29, and a gap of over a second leaves positions too far apart to tell. So version 2 does two narrower things:

1. **It joins tracks only where one plainly continues the other** (`join_track_fragments`): within 6 frames, within 0.1 box heights, at about the same size. This fixes an identity switch.
2. **It estimates a hidden entry** (`hidden_entry`): a track that starts just past the line, moving along the lane, is followed back to the line. It can only become a candidate if the signal was red for the whole time from 13 frames before the estimated entry until the vehicle was seen. Otherwise it is `not_evaluable_signal`. The event records `entry_estimated`, so a reviewer knows that the crossing itself was not seen.


In [27]:

V2_DIR = OUTPUT_DIR / "rule_v2"
v2_run = detect_red_light_video(VIDEO, V2_DIR, PROJECT_ROOT / "configs" / "scenes" / "rlr_occasions.json", rule=RULE_V2)

v1_keys = {(e["clip"], e["trigger_frame"]) for e in events}
print(f"candidates: version 1 {len(events)}, version 2 {len(v2_run['events'])}")
for event in v2_run["events"]:
    if (event["clip"], event["trigger_frame"]) not in v1_keys:
        print(f"  new: clip {event['clip']}, track {event['track_id']}, frame {event['trigger_frame']}, estimated entry: {event['evidence']['entry_estimated']}")
print("estimated entries among all crossing records:", sum(r["entry_estimated"] for records in v2_run["crossings"].values() for r in records))

print(f"\n{'clips':<14}{'version':>8}{'TP':>4}{'FP':>4}{'ignored':>9}{'FN':>4}{'precision':>11}{'recall':>8}")
for title, names in (("evaluation", EVALUATION), ("development", DEVELOPMENT)):
    for version, crossings in ((1, results), (2, v2_run["crossings"])):
        total = Counter()
        for name in names:
            outcome = score_clip(name, crossings[name])
            total.update(TP=len(outcome["true_positives"]), FP=len(outcome["false_positives"]), ignored=len(outcome["ignored"]), FN=len(outcome["false_negatives"]))
        tp, fp, fn = total["TP"], total["FP"], total["FN"]
        print(f"{title:<14}{version:>8}{tp:>4}{fp:>4}{total['ignored']:>9}{fn:>4}{tp / (tp + fp) if tp + fp else float('nan'):>11.2f}{tp / (tp + fn) if tp + fn else float('nan'):>8.2f}")

manifest = json.loads(v2_run["clips_manifest"].read_text())
assert len(manifest["clips"]) == len(v2_run["events"]) and all(e["rule"]["version"] == 2 for e in v2_run["events"])
assert all(e["rule"]["version"] == 1 for e in events)


candidates: version 1 16, version 2 17
  new: clip M, track 0+1, frame 3834, estimated entry: False
estimated entries among all crossing records: 0

clips          version  TP  FP  ignored  FN  precision  recall
evaluation           1  13   0        1   1       1.00    0.93
evaluation           2  14   0        1   0       1.00    1.00
development          1   2   0        0   5       1.00    0.29
development          2   2   0        0   5       1.00    0.29


## Results

All numbers come from the outputs above. The rule did not change after the evaluation clips were labelled or run.

| | Evaluation clips (15) | Development clips (3) | All 18 clips |
|---|---|---|---|
| **Precision** | 13/13 = 1.00 (0.77–1.00) | 2/2 = 1.00 (0.34–1.00) | 15/15 = 1.00 (0.80–1.00) |
| **Recall** | 13/14 = 0.93 (0.69–0.99) | 2/7 = 0.29 (0.08–0.64) | 15/21 = 0.71 (0.50–0.86) |

Intervals are 95% Wilson intervals. One further candidate on the evaluation clips matched a label marked `uncertain` and is excluded.

**These scores are version 1, the rule that was frozen.** Version 2, in section 13, fixes the miss M1 by joining its two tracks, and on these clips changes nothing else: 14 of 14 on the evaluation clips, with no false positive. M1 was known when version 2 was written, so that is not a fair test, and only the table below counts as one.

**Which recall to quote.** 0.93 holds only for footage with colour. Clip H, the one entirely monochrome clip, happens to be a development clip, and the rule declines to judge it. Across all 18 clips, recall is 0.71.

**Signal reading.** Across 523 scored readings, the reader was **never wrong**, but it did not always answer:

- Through heads, leaving out clip H: 225 of 228 answered, and the other 3 are monochrome frames at the start of clip I.
- Left head at night: 166 of 169, with the same 3 frames unanswered.
- Left head by day: 1 of 62. That head reads too dim in daylight.
- Monochrome clip H: 0 of 64, as designed.

**What the rule adds over a trivial one.** Flagging any crossing on red gives precision 0.78 and recall 1.00 (14 TP, 4 FP, 6 ignored). The rule's other conditions remove all 4 false positives, 2 vehicles stopped at the line and 2 tracks at the image border, at the cost of one missed violation.

**The signal controls the outcome.** Forcing the state to green, yellow, unknown or conflict gives 0 candidates. Forcing red gives 15, the extra one being the car that really crossed on yellow.

**Sensitivity.** The table needs careful reading, because some of its errors come from the matching and not from the rule:

- **Stop line moved by ±20 px, entry offset 0.20 or 0.40:** the rule keeps finding the same violations. The false positive and false negative in those rows are one vehicle counted twice: `D t12` is the motorcycle of D1, and `G t1` is the vehicle of G1. Moving the entry point shifts their position until they pair with the wrong label or fall outside the limit. Moving the line 20 px up also recovers M1, and moving it down drops the uncertain R-u1.
- **Pass margin 0.75:** really loses C1 and C2.
- **Signal read 0.5 s late:** really loses C1 and N2, which entered 0.4 s after the change. 5 of the 13 true positives entered within 0.7 s of it, so **timing is the most sensitive part of the result**.

**Logic test with a synthetic signal.** All 6 crossings behaved as designed: 3 on green and 1 on yellow were not flagged, 1 on red was flagged, and a truck that stopped on red was not flagged. This shows that the logic works on other footage, and says nothing about real violations.

**Clips.** Each of the 16 candidates has its own video in `violation_clips/`, 39.2 MB in total, with `manifest.json` giving the vehicle's box in each frame. Every clip has the exact frame count, matches the source at the moment of entry (PSNR 42.4–43.7 dB), and is as sharp as the source (ratio 0.99–1.01). The same step also ran on the synthetic-signal video, at another size and frame rate.

**One function.** `detect_red_light_video` joins every step, from reading the video to a video of the results, the clips and a file of events (section 12). It reproduces all 16 candidates and the same scores.

**Reproducibility.** The tracks of all 18 clips are identical to those of an earlier, separate run.

## False positives and false negatives

- **False positives:** none on any clip.
- **Ignored:** R-u1, a blue sedan with its front already across the line when the clip starts. The signal at its entry cannot be seen, so a reviewer must decide. It stays a `pending` candidate.
- **False negative on the evaluation clips, M1:** the tracker switched IDs at f3855. The first track ended with the pickup's rear 16 px past the line, short of the half box height required, and the second started beyond the line. The rule reports `not_evaluable_ended` for it, so a reviewer can still see it.
- **False negatives on the development clips:** H1–H4 are `not_evaluable_signal` in the monochrome clip. H5 has no crossing record at all.

## Limitations

**The labels are not independent ground truth.**

- They were made by the AI assistant that wrote the rule, and **no person has reviewed them**.
- While labelling, the annotator estimated a vehicle's front with the rule's own formula. So the close agreement on timing (median 2 frames, maximum 6) does not validate that formula.
- The annotator knew the source is a compilation of violations.

**The test is easy, because the footage was selected for violations.**

- 24 of the 25 crossings on the evaluation clips happened on red, 1 on yellow, and **none on green**. Each clip has a median of 3 tracked vehicles.
- Only 3 of those crossings were clear negatives: 2 stopped at the line and 1 on yellow. Nearly any crossing here is a violation, so precision of 1.00 **must not be expected on ordinary traffic**, where most crossings are legal.
- The sample is small: the lower bound of the precision interval is 0.77.

**Parts of the rule that real data did not test:**

- **Right turn on red:** this verdict never occurred, so that logic is covered by unit tests only.
- **The right-turn lane:** 7 of the 8 crossings not judged are in this lane, nearly all with boxes at the image border.
- **Separate signal groups:** only two real crossings used the left group, C2 and a car stopped at the line, and the through signal was red both times. The assumption about which head governs which lane has never decided an outcome.

**Conditions the rule cannot judge:** monochrome footage, and the left lane by day.

**Plates may not be readable in many of these clips.** In the sheet of largest views, the plate area is distinct in the 4 daytime clips (A, B, D, R). In the 12 night clips, fast vehicles are motion-blurred and tail lights flare. This loss is in the source, since the clips match it, so Notebook 06 should expect little from them. No plate was read here, so this is a judgement by eye.

**Clips exist only for candidates.** A labelled violation the rule declines to judge, such as M1 or those in clip H, gets a review clip but no violation clip, so it never reaches plate reading.

**Other limits:**

- **A new video needs its geometry first.** The stop line and signal positions are declared by hand for each camera framing, so a video cannot yet be analysed without that step.
- **One junction, one viewpoint.** Nothing here shows how the rule behaves at another site.
- **The stop line is inferred** from the ends of the lane markings, because the paint is faint or absent.
- **The matching criterion has a flaw.** It takes the closest label in time, so a candidate can pair with a nearby vehicle's label. D1 avoided this only because the stopped car's label is 210 px away, against a limit of 200 px. The headline numbers are the same with limits of 200 and 150 px.
- **Each track is judged once.** Only its first crossing counts, so a track that absorbs a second vehicle hides that vehicle.
- **Footage quality.** This is a re-encoded screen recording with video-player overlays. Clip N contains a cut, and the tracker was not reset there.

## Conclusion

**Completion criterion** — *the selected rule reliably produces candidate events in representative scenarios and does not silently treat missing traffic conditions as confirmed violations*:

- **Missing conditions: met.** An unreadable signal, a box at the border, an overlay or a track that ends all give a `not_evaluable` status, every crossing is recorded, and every event stays `pending`.
- **Producing candidates: met on this footage, not shown beyond it.** It finds 13 of 14 labelled violations with colour, with no false positive, on clips not used to shape it. Those labels still await review by a person, and the footage is not representative of ordinary traffic.

**Readiness for integration.** The rule, the clip cutter and `detect_red_light_video` are in `src/traffic_violation_detection/` (`red_light.py`, `evidence.py`) with tests. `violation_clips/manifest.json` is the input to Notebook 06. Events follow the structure of Notebook 05, and Notebook 07 still has to settle the shared schema.

**Next steps.**

- **Human review of the labels** is the most valuable next step: `review/review_sheet.csv` lists 30 cases, each with an annotated clip, and any change alters the scores.
- **Notebook 06 — license plate recognition** can take `violation_clips/manifest.json` as its input, and `traffic_video_modified.mp4` for plates that are readable, keeping that video's results out of any real event.
- **Improvements not made here:** reading the lit lamp's position in monochrome footage (clip H), and labelling vehicles with boxes for a sturdier match. Testing these and version 2 fairly needs **new footage**, ideally of ordinary traffic that includes many legal crossings.